# Landsat Land Surface Temperature time series analysis. 

The Google Earth Engine (GEE) code uses Landsat Collection 2 - Land Surface Temperature (LST) and harmonic modeling to produce maps
of:
1) Mean annual LST (MALST)
2) Amplitude
3) Phase shift
4) LST Trend
5) RMSE
6) Number of observations (counts)

Landsat LST originate from:
- USGS Landsat 5 Level 2, Collection 2, Tier 1
- USGS Landsat 7 Level 2, Collection 2, Tier 1
- USGS Landsat 8 Level 2, Collection 2, Tier 1


In [4]:
import os
import ee
#connect to gee python api
#ee.Authenticate()
ee.Initialize()
import geemap
import pandas as pd
import geopandas as gpd
import math

In [5]:
# scale factor and cloud masking
def applyScaleFactorsL457(image):
    #apply scale and offset for Landsat C2 bands
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2) #SR=SurfaceReflection
    thermalBand = image.select('ST_B6').multiply(0.00341802).add(149.0) #ST=SurfaceTemperature
    emissBand = image.select('ST_EMIS').multiply(0.0001)
    return image.addBands(opticalBands, overwrite = True).addBands(thermalBand, overwrite = True).addBands(emissBand, overwrite = True)

def applyScaleFactorsL8(image):
    #apply scale and offset for Landsat C2 bands
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2) #SR=SurfaceReflection
    thermalBand = image.select('ST_B10').multiply(0.00341802).add(149.0) #ST=SurfaceTemperature
    emissBand = image.select('ST_EMIS').multiply(0.0001) # emissivity band
    return image.addBands(opticalBands, overwrite = True).addBands(thermalBand, overwrite = True).addBands(emissBand, overwrite = True)

def maskcloudL457(image):
    
    stThMin = ee.Number(-60+273.15)
    stThMax = ee.Number(60+273.15)

    #Bits 3 and 4 are cloud and cloud shadow, respectively.
    #Bits 9 and 11 indicate at least medium confidence for cloud and cloud shadow
    qa = image.select('QA_PIXEL')
    cloud = qa.bitwiseAnd(1<<3).eq(0)
    cloudconf = qa.bitwiseAnd(1<<9).eq(0)
    
    mask2 = qa.mask()

    image_masked = image.updateMask(mask2)
    image_masked = image_masked.updateMask(cloud)
    image_masked = image_masked.updateMask(cloudconf)
    image_masked = image_masked.updateMask(image_masked.gte(stThMin).And(image_masked.lte(stThMax))) #redundant
    image_masked = image_masked.select(['ST_B6','ST_EMIS'], ['SurfT','Emis'])
    
    return image_masked.copyProperties(image, ["system:time_start"]) 

def maskcloudL8(image):
    
    stThMin = ee.Number(-60+273.15)
    stThMax = ee.Number(60+273.15)

    #Bits 3 and 4 are cloud and cloud shadow, respectively.
    #Bits 9 and 11 indicate at least medium confidence for cloud and cloud shadow
    qa = image.select('QA_PIXEL')
    cloud = qa.bitwiseAnd(1<<3).eq(0)
    cloudconf = qa.bitwiseAnd(1<<9).eq(0)
    
    mask2 = qa.mask()

    image_masked = image.updateMask(mask2)
    image_masked = image_masked.updateMask(cloud)
    image_masked = image_masked.updateMask(cloudconf)
    image_masked = image_masked.updateMask(image_masked.gte(stThMin).And(image_masked.lte(stThMax))) #redundant
    image_masked = image_masked.select(['ST_B10','ST_EMIS'], ['SurfT','Emis'])
    
    return image_masked.copyProperties(image, ["system:time_start"]) 


# Remove any pixels that are contained in the next image.
def mask_overlap(image):
    _next = ee.Image(image.get('match'))
    return ee.Image(image).updateMask(_next.mask().Not())

# A filter for images in the same path, differing by less than 100 seconds, one direction in time.
alongtrack_overlapfilter = ee.Filter.And(ee.Filter.equals("WRS_PATH", None, "WRS_PATH"), 
                                         ee.Filter.greaterThan("system:time_start", None, "system:time_start"),
                                         ee.Filter.maxDifference(100000, "system:time_start", None, "system:time_start")
                                        )
# function for harmonic modeling
def addDepVar(image):
    return image.select('SurfT')

def addIndVar(image):
    date = ee.Date(image.get('system:time_start'))
    years = date.difference(ee.Date('1970-01-01'), 'year')
    
    timeRadians = ee.Image.constant(ee.List([1])).multiply(years).multiply(2*math.pi).float() # 1 = one harmonic oscilation
    
    con = ee.Image.constant(1).clip(image.geometry()).updateMask(image.mask().select(0)) # ee.constant create image without bounds, therfore clip and mask
    
    cos = ee.Image(timeRadians).cos().clip(image.geometry()).updateMask(image.mask().select(0))
    sin = ee.Image(timeRadians).sin().clip(image.geometry()).updateMask(image.mask().select(0))
    
    yrs = ee.Image.constant(years).float().clip(image.geometry()).updateMask(image.mask().select(0))
    
    return image.addBands(con).addBands(yrs.rename('t')).addBands(cos.rename(coscoefnames)).addBands(sin.rename(sincoefnames))

def modelling(image):
    fit = image.select(harmonicIndependents).multiply(harmonicTrendCoefficients).reduce('sum').rename('fitted')
    error = image.select('SurfT').subtract(fit.select('fitted')).pow(2).rename('error')   
    return image.addBands(image.select('SurfT')).addBands(fit).addBands(error)

def correction(image):
    date = ee.Date(image.get(timeField))
    time = date.difference(ee.Date('1970-01-01'), 'year')
    
    timeRadians = time.multiply(2 * math.pi).float()

    pred = _b0.add(_b1.multiply(time)).add(_b2.multiply(timeRadians.cos())).add(_b3.multiply(timeRadians.sin())).rename('modelled')
    mod = pred.select('modelled')
    obs = image.select('SurfT')
    
    res = ee.Image((obs.subtract(mod)).abs())
    res_mask = res.gte(30)
    return image.updateMask(res_mask.Not()) 

In [6]:
def ExportEEdata_image(image, basename, folder, exportTo='drive'):
    '''
    image (ee.Image)
    basename (str) 
    folder (str)
    toAsset (std): 'drive' or 'asset' 
    coordinates (list of tuples or None) e.g. [(lat, lon), (lat, lon), ...]
    '''
    
    if exportTo=='drive':
        print('Image exported to Google Drive: ' + folder)
        
        task = ee.batch.Export.image.toDrive(**{
            'image': image,
            'description': basename,
            'folder': folder,
            'scale': 30,
            'maxPixels': 1e12, 
            'crs':'EPSG:32632',
            'fileFormat':'GeoTIFF'})   
        task.start()
        print('state: ', task.status()['state'])
    
        
    elif exportTo=='asset':
        print('Image exported to GEE assets: ' 'projects/dtgoek/assets/'+ basename)
        
        task = ee.batch.Export.image.toAsset(**{
            'image': image,
            'description': basename,
            'assetId':'projects/dtgoek/assets/'+basename,
            'scale': 30, 
            'maxPixels': 1e12})
        task.start()
        print('state: ', task.status()['state'])
        
    else:
        print('exportTo must be "drive" or "asset" ')

## Settings:

In [7]:
# if additional_cloud_filter = False, beta coefficients musst be calculated in a previous step and uploaded as asset to gee
additional_cloud_filter = False

if (additional_cloud_filter):
    _b0 = ee.Image('projects/dtgoek/assets/b0_largeExt_staticThresh').divide(100) #MALST
    _b1 = ee.Image('projects/dtgoek/assets/b1_largeExt_staticThresh').divide(1000) #TREND
    _b2 = ee.Image('projects/dtgoek/assets/b2_largeExt_staticThresh').divide(100).subtract(100)
    _b3 = ee.Image('projects/dtgoek/assets/b3_largeExt_staticThresh').divide(100).subtract(100)
    
start = '1984-01-01' 
end = '2022-12-31'
sensor = 'L578' # full Landsat time series OR 'l5', 'l7', 'l8', 'l57', 'l58', etc

# region of interest, Swiss Alps
xmin = 5.41
xmax = 10.97
ymin = 45.34
ymax = 47.97

bounds = ee.Geometry.Rectangle(ee.Number(xmin),ee.Number(ymin),ee.Number(xmax),ee.Number(ymax)) 

filename_date = '_' + start + '_' + end
export_destination = 'drive' #or asset

export_malst = False
export_trend = False
export_amplitude = False
export_phase = False
export_rmse = False
export_counts = False

## Landsat image collection (cloud masked, along track overlap masked)

In [8]:
# create Landsat ImageCollection
L5_data = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2').filterDate(start, end).filterBounds(bounds)
L7_data = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2').filterDate(start, '2020-01-01').filterBounds(bounds)
L8_data = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate(start, end).filterBounds(bounds)

# apply sclae factors
L5_data = L5_data.map(applyScaleFactorsL457)
L7_data = L7_data.map(applyScaleFactorsL457)
L8_data = L8_data.map(applyScaleFactorsL8)


L5_data = L5_data.map(maskcloudL457)
L7_data = L7_data.map(maskcloudL457)
L8_data = L8_data.map(maskcloudL8)


if sensor.lower() == 'l5':
    lscomp=ee.ImageCollection(L5_data)
    
if sensor.lower() == 'l7':
    lscomp=ee.ImageCollection(L7_data)

if sensor.lower() == 'l8':
    lscomp=ee.ImageCollection(L8_data)
    
if sensor.lower() == 'l57':
    lscomp=ee.ImageCollection(L7_data.merge(L5_data))
    
if sensor.lower() == 'l78':
    lscomp=ee.ImageCollection(L8_data.merge(L7_data))
    
if sensor.lower() == 'l58':
    lscomp=ee.ImageCollection(L8_data.merge(L5_data))
    
if sensor.lower() == 'l578':
    lscomp=ee.ImageCollection(L8_data.merge(L7_data).merge(L5_data))  
    
lscomp = lscomp.sort('system:time_start', True)

print('n Landsat scenes: ', lscomp.size().getInfo())

joined = ee.ImageCollection(ee.Join.saveBest("match", "measure", True).apply(lscomp, lscomp, alongtrack_overlapfilter))

# Split the join into those with matches and those without
noMatch = ee.ImageCollection(joined.filter(ee.Filter.notNull(['match']).Not()))
hasMatch = ee.ImageCollection(joined.filter(ee.Filter.notNull(['match'])))

lscomp = hasMatch.map(mask_overlap)

lscomp = lscomp.filterBounds(bounds)

n Landsat scenes:  16068


## Harmonic model regression

In [11]:
sincoefnames = ee.List(['sin'])
coscoefnames = ee.List(['cos'])

timevar = ee.String('t')
constant = ee.String('constant')
dependent = ee.String('SurfT')

timeField = 'system:time_start'

harmonicIndependents = ee.List([constant]).add(timevar).add(coscoefnames).add(sincoefnames).flatten()
#print(harmonicIndependents.getInfo())

filteredLandsat = lscomp.select('SurfT')
filteredLandsat = filteredLandsat.map(addIndVar)

In [12]:
def modelling(image):
    fit = image.select(harmonicIndependents).multiply(harmonicTrendCoefficients).reduce('sum').rename('fitted')
    error = image.select('SurfT').subtract(fit.select('fitted')).pow(2).rename('error')   
    return image.addBands(image.select('SurfT')).addBands(fit).addBands(error)

In [14]:
# select bands for regression
# https://developers.google.com/earth-engine/apidocs/ee-reducer-linearregression

harmonicTrend = filteredLandsat.select(harmonicIndependents.add(dependent))
harmonicTrend = harmonicTrend.reduce(ee.Reducer.linearRegression(numX=4, numY=1))

harmonicTrendCoefficients = harmonicTrend.select('coefficients').arrayProject([0]).arrayFlatten([harmonicIndependents])

b0 = harmonicTrendCoefficients.select('constant')
b1 = harmonicTrendCoefficients.select('t')
b2 = harmonicTrendCoefficients.select('cos1')
b3 = harmonicTrendCoefficients.select('sin1')

fittedHarmonic = filteredLandsat.map(modelling)

LST_trend = b1.multiply(1000).clip(bounds)
MALST = b0.multiply(100).int16().clip(bounds)
amplitude = b3.hypot(b2).multiply(1000).int16().clip(bounds)
phase = b3.atan2(b2).unitScale(-math.pi, math.pi).multiply(10000).uint16().clip(bounds)
counts = ee.Image(filteredLandsat.select('SurfT').count()).uint16().clip(bounds)
rmse = fittedHarmonic.select('error').reduce(ee.Reducer.mean()).sqrt().uint16().multiply(100).uint16().clip(bounds)

if (additional_cloud_filter==False):
    print('Export without residual-threshold cloud masking')
    

    if (export_malst):
        ExportEEdata_image(image = MALST, 
                           basename = 'MALST' + filename_date, 
                           folder = 'EE_2023_11_LST_timeseries_regression', 
                           exportTo = export_destination)
    if (export_trend):
        ExportEEdata_image(image = LST_trend.int16(), 
                           basename = 'LSTtrend' + filename_date, 
                           folder = 'EE_2023_11_LST_timeseries_regression', 
                           exportTo = export_destination)
    if (export_amplitude):
        ExportEEdata_image(image = amplitude, 
                           basename = 'amplitude' + filename_date, 
                           folder = 'EE_2023_11_LST_timeseries_regression', 
                           exportTo = export_destination)
    if (export_phase):
        ExportEEdata_image(image = phase, 
                           basename = 'phase' + filename_date, 
                           folder = 'EE_2023_11_LST_timeseries_regression', 
                           exportTo = export_destination)
    if (export_rmse):
        ExportEEdata_image(image = rmse, 
                           basename = 'rmse' + filename_date, 
                           folder = 'EE_2023_11_LST_timeseries_regression', 
                           exportTo = export_destination)
    if (export_counts):
        ExportEEdata_image(image = counts, 
                           basename = 'counts' + filename_date, 
                           folder = 'EE_2023_11_LST_timeseries_regression', 
                           exportTo = export_destination)
        

Export without residual-threshold cloud masking


In [15]:
if (additional_cloud_filter):
    
    filteredLandsat = filteredLandsat.map(correction)
    #filteredLandsat = filteredLandsat.map(addDepVar)

    harmonicTrend = filteredLandsat.select(harmonicIndependents.add(dependent))
    harmonicTrend = harmonicTrend.reduce(ee.Reducer.linearRegression(numX=4, numY=1))

    harmonicTrendCoefficients = harmonicTrend.select('coefficients').arrayProject([0]).arrayFlatten([harmonicIndependents])

    b0 = harmonicTrendCoefficients.select('constant')
    b1 = harmonicTrendCoefficients.select('t')
    b2 = harmonicTrendCoefficients.select('cos1')
    b3 = harmonicTrendCoefficients.select('sin1')

    fittedHarmonic = filteredLandsat.map(modelling)

    LST_trend = b1.multiply(1000).clip(bounds_export)
    MALST = b0.multiply(100).int16().clip(bounds_export)
    amplitude = b3.hypot(b2).multiply(1000).int16().clip(bounds_export)
    phase = b3.atan2(b2).unitScale(-math.pi, math.pi).multiply(10000).uint16().clip(bounds_export)
    counts = ee.Image(filteredLandsat.select('SurfT').count()).uint16().clip(bounds_export)
    rmse = fittedHarmonic.select('error').reduce(ee.Reducer.mean()).sqrt().uint16().multiply(100).uint16().clip(bounds_export)

    print('Export with residual-threshold cloud masking')
    if (export_malst):
        ExportEEdata_image(image = MALST, 
                           basename = 'MALST' + filename_date, 
                           folder = 'EE_2023_11_LST_timeseries_regression', 
                           exportTo = export_destination)
    if (export_trend):
        ExportEEdata_image(image = LST_trend.int16(), 
                           basename = 'LSTtrend' + filename_date, 
                           folder = 'EE_2023_11_LST_timeseries_regression', 
                           exportTo = export_destination)
    if (export_amplitude):
        ExportEEdata_image(image = amplitude, 
                           basename = 'amplitude' + filename_date, 
                           folder = 'EE_2023_11_LST_timeseries_regression', 
                           exportTo = export_destination)
    if (export_phase):
        ExportEEdata_image(image = phase, 
                           basename = 'phase' + filename_date, 
                           folder = 'EE_2023_11_LST_timeseries_regression', 
                           exportTo = export_destination)
    if (export_rmse):
        ExportEEdata_image(image = rmse, 
                           basename = 'rmse' + filename_date, 
                           folder = 'EE_2023_11_LST_timeseries_regression', 
                           exportTo = export_destination)
    if (export_counts):
        ExportEEdata_image(image = counts, 
                           basename = 'counts' + filename_date, 
                           folder = 'EE_2023_11_LST_timeseries_regression', 
                           exportTo = export_destination)
        